# Two-dimensional triangular-mesh deposition

Ten beams are traced through a cylindrical hydro grid. Their sheet-resolved affine source is integrated exactly onto a separate circular triangle mesh by the canonical JAX overlap kernels.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.tri as mtri
import numpy as np

from pyGATH.fields import (
    build_circular_deposition_mesh_from_grid,
    deposit_simplicial_power_to_mesh,
    simplicialise_sheet_fields,
)
from pyGATH.io import load_simulation_config

root = Path.cwd().resolve()
if root.name == "examples":
    root = root.parent
simulation = load_simulation_config(
    root / "configs/example_configs/paper_s64_ten_beam_cylindrical_deposition.toml"
)

In [ ]:
with simulation.reporting():
    grid = simulation.build_grid()
    beams = simulation.load_beams()
    initial_rays = simulation.initialize_rays(grid, beams=beams)
    trace = simulation.trace_rays(initial_rays, grid)
    source = simplicialise_sheet_fields(
        trace.sheet_fields, dimension=2, fields="inverse_brems_deposition"
    )
    target = build_circular_deposition_mesh_from_grid(grid, maximum_angular_cells=80)
    deposition = deposit_simplicial_power_to_mesh(source, target)
print(f"{source.mesh.nsimplices:,} source triangles per sheet")
print(f"{target.ncells:,} target triangles")
print(f"conservation error={deposition.conservation_error:.3e} W")

In [ ]:
triangulation = mtri.Triangulation(
    target.vertex_positions[:, 0],
    target.vertex_positions[:, 1],
    target.simplex_connectivity,
)
figure, axis = plt.subplots(figsize=(7, 6))
image = axis.tripcolor(
    triangulation, facecolors=np.asarray(deposition.power_density), shading="flat"
)
axis.set_aspect("equal")
axis.set_xlabel("x [m]")
axis.set_ylabel("y [m]")
figure.colorbar(image, ax=axis, label="power density [W/m^3]")